# Milestone 4: Multi-Hop Chain Detection + Explainability

## 1. Executive Context & Motivation

In **Milestone 3**, we demonstrated that GraphSAGE achieves a **2.28x lift** in PR-AUC over tabular XGBoost (0.1884 vs 0.0834) and a **15.3x lift** over random baseline. However, GNNs present two major limitations in real-world AML compliance operations:

1. **Depth vs Over-Smoothing:** Stacking 4+ GNN layers causes node embeddings to converge and lose discriminative power. Yet real-world layering chains span 3 to 5 hops ($A \to B \to C \to D$). Structural pattern search (temporal motif matching) solves this without risking over-smoothing.
2. **The Explainability Imperative:** A financial institution cannot legally file a **Suspicious Activity Report (SAR)** or freeze an account based purely on a raw probability score (e.g. `0.7004`). Compliance analysts and regulators require **interpretable rationale**: *Which transactions? Which counterparties? What behavioral flags driven by what features?*

**Milestone 4 delivers:**
- **Algorithmic Temporal Motif Search (`TemporalChainDetector`):** Identifies multi-hop linear chains ($A \to B \to C \to D$) with chronological progression and amount preservation/decay.
- **GNNExplainer (`MuleExplainer`):** Identifies the minimal computation subgraph and feature attributions driving the model's prediction.
- **Analyst Risk Cards (`AccountRiskReporter`):** Produces human-readable, SAR-ready compliance reports and ego-network risk visualizations.

## 2. Setup & Component Initialization

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.gnn_data import build_pyg_data
from src.gnn_model import GraphSAGEClassifier
from src.chain_detector import TemporalChainDetector
from src.explainer import MuleExplainer
from src.risk_report import AccountRiskReporter

# Load Data
df = pd.read_csv('../data/processed/processed_df.csv')
features_df = pd.read_csv('../data/processed/account_features.csv')
split_df = pd.read_csv('../data/processed/split_accounts.csv')
data = build_pyg_data(df, features_df, split_df)

# Load Trained GraphSAGE
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GraphSAGEClassifier(in_dim=8, hidden_dim=64, num_layers=2).to(device)
model.load_state_dict(torch.load('../outputs/models/graphsage.pt', map_location=device))
model.eval()

# Initialize Milestone 4 Modules
detector = TemporalChainDetector(df, max_decay=0.25)
explainer = MuleExplainer(model, data, threshold=0.5254, epochs=60, device=device)
reporter = AccountRiskReporter(detector, explainer)
print('All Milestone 4 modules successfully initialized on', device)

## 3. Multi-Hop Temporal Chain Detection (Motif Search)

Money laundering layering relies on sequential fund transfers with small fee deductions at each intermediary hop.
Below we search for multi-hop chains and inspect a verified layering sequence.

In [2]:
# Inspect multi-hop chains passing through intermediary account 811C2F680
target_mule = '811C2F680'
profile = detector.get_account_chain_profile(target_mule)
print(f"Account: {target_mule}")
print(f"Role: {profile['role'].upper()}")
print(f"Total Chains Found: {profile['num_chains_found']}")
print(f"Max Chain Length: {profile['max_chain_hops']} hops\n")

print("Sample Discovered Chains:")
for i, c in enumerate(profile['sample_chains'][:3], 1):
    path_str = ' -> '.join(c['accounts'])
    print(f"  [{i}] {path_str}")
    print(f"      Inflow: ${c['initial_amount']:,.2f} | Outflow: ${c['final_amount']:,.2f} | Retention: {c['amount_retention']*100:.1f}%")
    print(f"      Duration: {c['duration_hours']:.1f} hours")

## 4. GNNExplainer: Attributing Model Predictions

We now apply GNNExplainer to `811C2F680` to discover which local features and edges caused GraphSAGE to assign a high fraud probability.

In [3]:
gnn_result = explainer.explain_node(target_mule)
print(f"GNN Fraud Probability: {gnn_result['gnn_prob']*100:.2f}%")
print(f"Decision Status: {'FLAGGED AS MONEY MULE' if gnn_result['is_flagged'] else 'NORMAL'}\n")

print("Top Feature Attributions:")
for f in gnn_result['top_features']:
    print(f"  - {f['feature']:<25}: {f['score']*100:5.1f}% contribution")

print("\nTop Influential Transaction Flows:")
for e in gnn_result['top_edges'][:4]:
    print(f"  - {e['direction'].upper():<8} with {e['counterparty']:<12} (score: {e['importance_score']:.3f})")

## 5. End-to-End Compliance Deliverable: Analyst Risk Card

Synthesizing GNN probability, multi-hop motif topology, and feature/edge attributions into an AML Analyst Risk Card and Ego-Network visualization.

In [4]:
report = reporter.analyze_account(target_mule)
card_text = reporter.format_risk_card(report)
print(card_text)

plot_path = reporter.plot_risk_subgraph(report, output_path='../outputs/figures/risk_cards/811C2F680_subgraph.png')
display(Image(filename=plot_path))

## 6. Comparative Case Studies

To demonstrate robustness, we compare:
1. **Confirmed Money Mule (True Positive):** Account `811C2F680`
2. **Legitimate Account (True Negative):** High-volume counterparty with zero laundering activity.

In [5]:
# Find a prominent non-laundering account
legit_candidates = df[df['Is Laundering'] == 0]['from_Account'].value_counts()
legit_acc = str(legit_candidates.index[0])

legit_report = reporter.analyze_account(legit_acc)
print(f"Legitimate Account: {legit_acc}")
print(f"GNN Mule Probability: {legit_report['gnn_explanation']['gnn_prob']*100:.2f}%")
print(f"Risk Tier: {legit_report['risk_tier']}")
print(f"In Active Laundering Chain: {legit_report['chain_profile']['is_in_chain']}")
print(f"Recommended Action: {legit_report['recommended_action']}")

## 7. Milestone 4 Summary & Takeaways

1. **Complementary Fraud Detection:** GNNs identify structural node anomalies (rapid velocity, degree dispersion), while algorithmic temporal motif search traces multi-hop layering chains ($A \to B \to C \to D$) across time windows and amount decay without over-smoothing.
2. **Investigator-Ready Explainability:** `GNNExplainer` transforms opaque node embeddings into quantifiable feature importance percentages and counterparty rankings.
3. **Regulatory Readiness:** Analyst Risk Cards synthesize mathematical scores into actionable plain-language narratives suitable for immediate SAR filing.

### Next: Milestone 5 (Scale + Interactive Demo)
- Re-implement feature engineering in **PySpark + GraphFrames** for HI-Medium and HI-Large scales.
- Build an interactive **Streamlit dashboard** allowing compliance officers to search any account ID, view interactive ego-subgraphs, and generate instant Risk Cards.